In [3]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [7]:
#example tqdm
for i in tqdm(range(100)):
    x = 1 + 2
    time.sleep(0.1)

100%|██████████| 100/100 [00:10<00:00,  9.86it/s]


In [11]:
df = pd.read_csv("Surgical-deepnet.csv")
df.head()

,bmi,Age,asa_status,baseline_cancer,baseline_charlson,baseline_cvd,baseline_dementia,baseline_diabetes,baseline_digestive,baseline_osteoart,...,complication_rsi,dow,gender,hour,month,moonphase,mort30,mortality_rsi,race,complication
0,19.31,59.2,1,1,0,0,0,0,0,0,...,-0.57,3,0,7.63,6,1,0,-0.43,1,0
1,18.73,59.1,0,0,0,0,0,0,0,0,...,0.21,0,0,12.93,0,1,0,-0.41,1,0
2,21.85,59.0,0,0,0,0,0,0,0,0,...,0.00,2,0,7.68,5,3,0,0.08,1,0
3,18.49,59.0,1,0,1,0,0,1,1,0,...,-0.65,2,1,7.58,4,3,0,-0.32,1,0
4,19.70,59.0,1,0,0,0,0,0,0,0,...,0.00,0,0,7.88,11,0,0,0.00,1,0


In [12]:
df.isnull().sum()

bmi                    0
Age                    0
asa_status             0
baseline_cancer        0
baseline_charlson      0
baseline_cvd           0
baseline_dementia      0
baseline_diabetes      0
baseline_digestive     0
baseline_osteoart      0
baseline_psych         0
baseline_pulmonary     0
ahrq_ccs               0
ccsComplicationRate    0
ccsMort30Rate          0
complication_rsi       0
dow                    0
gender                 0
hour                   0
month                  0
moonphase              0
mort30                 0
mortality_rsi          0
race                   0
complication           0
dtype: int64

In [13]:
X = df.drop("complication",axis=1).copy().values
Y = df["complication"].copy().values

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2)
X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

((11708, 24), (11708,), (2927, 24), (2927,))

In [32]:
class Perceptron:
    def __init__(self, learning_rate,input_length):
        self.learning_rate = learning_rate
        self.weights = np.random.rand(input_length)
        self.bias = np.random.rand(1)

    def activation(self,x,function):
        if function == "sigmoid":
            return 1 / (1 + np.exp(-x))
        elif function == "relu":
            return np.maximum(0, x)
        elif function == "tanh":
            return np.tanh(x)
        elif function == "linear":
            return x
        else:
            raise ValueError("Function not recognized")

    def fit(self, X_train, Y_train, epochs = 100):
        for epoch in tqdm(range(epochs)):
            for x,y in zip(X_train, Y_train):
                #forwarding
                y_pred = x @ self.weights + self.bias
                y_pred = self.activation(y_pred, "sigmoid")
                #back propagation
                eror = y - y_pred

                #updating
                self.weights += self.learning_rate * eror * x
                self.bias += self.learning_rate * eror

    def predict(self,X_test):
        Y_pred = []
        for x in X_test:
            y_pred = x @ self.weights + self.bias
            y_pred = self.activation(y_pred, "sigmoid")
            Y_pred.append(y_pred)
        return np.array(Y_pred)

    def calculate_loss(self,X_test,Y_test,metric):
        Y_pred = self.predict(X_test)
        if metric == "mse":
            return np.mean(np.square(Y_test - Y_pred))
        elif metric == "mae":
            return np.mean(np.abs(Y_test - Y_pred))
        elif metric == "rsme":
            return np.sqrt(np.mean(np.sqrt(Y_test - Y_pred)))
        else:
            raise ValueError("Metric not recognized")

    def calculate_accuracy(self,X_test,Y_test):
        Y_pred = self.predict(X_test)
        Y_pred = Y_pred.reshape(-1)
        Y_pred = np.where(Y_pred > 0.5, 1,0)
        accuracy = np.sum(Y_pred == Y_test) / len(Y_pred)
        return accuracy

    def evaluate(self, X_test, Y_test):
        loss = self.calculate_loss(X_test,Y_test,"mae")
        accuracy = self.calculate_accuracy(X_test,Y_test)
        return loss,accuracy

In [33]:
model = Perceptron(learning_rate=0.001,input_length=X_train.shape[1])
model.fit(X_train, Y_train, epochs = 256)

100%|██████████| 256/256 [00:52<00:00,  4.85it/s]


In [34]:
model.evaluate(X_test, Y_test)

(np.float64(0.3815791155512012), np.float64(0.7837376153057738))

In [35]:
Y_pred = model.predict(X_test)
Y_pred = Y_pred.reshape(-1)
Y_pred = np.where(Y_pred > 0.5, 1,0)

In [36]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test,Y_pred)
cm

array([[1919,  239],
       [ 394,  375]])